In [ ]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('omw-1.4')


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report


In [ ]:
# importing the dataset

df= pd.read_csv("/kaggle/input/cyberbullying-tweets-dataset/cyber_bullying_data/train_data.csv")
df1=pd.read_csv("/kaggle/input/cyberbullying-tweets-dataset/cyber_bullying_data/test_data.csv")

In [ ]:
df.drop(df[df['cyberbullying_type'] == 'not_cyberbullying'].index, inplace=True)

In [ ]:
df1.drop(df1[df1['cyberbullying_type'] == 'not_cyberbullying'].index, inplace=True)

In [ ]:
df.shape

In [ ]:
df1['cyberbullying_type'].value_counts()

In [ ]:
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download stopwords if not already downloaded
nltk.download('stopwords')

def preprocess_text(text):
    """Cleans text by lowercasing, removing stopwords, and applying stemming."""
    ps = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove non-alphabetic characters (keep spaces)
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Tokenize and remove stopwords, apply stemming
    words = text.split()
    words = [ps.stem(word) for word in words if word not in stop_words]
    
    # Join words back into a string
    return " ".join(words)

df = df.copy()
df1 = df1.copy()

# ✅ Use .loc to assign new column safely
df.loc[:, 'cleaned_text'] = df['tweet_text'].apply(preprocess_text)
df1.loc[:, 'cleaned_text'] = df1['tweet_text'].apply(preprocess_text)

In [ ]:
# Encoding the labels
label_mapping = {
    "other_cyberbullying": 0,
    "religion": 1,
    "age": 2,
    "gender": 3,
    "ethnicity": 4,
}


# Apply mapping
df["cyberbullying_encoded_type"] = df["cyberbullying_type"].map(label_mapping)
df1["cyberbullying_encoded_type"] = df1["cyberbullying_type"].map(label_mapping)

# lemmatization

lm = nltk.WordNetLemmatizer()

def text_lemmatization(text):
    text = [lm.lemmatize(word) for word in text]
    return text

df.loc[:, 'cleaned_text'] = df['tweet_text'].apply(preprocess_text)
df1.loc[:, 'cleaned_text'] = df1['tweet_text'].apply(preprocess_text)

In [ ]:
df

In [ ]:
# Splitting the data into train and test
X_train,y_train,X_test,y_test = df['cleaned_text'], df['cyberbullying_encoded_type'],df1["cleaned_text"],df1["cyberbullying_encoded_type"]

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
vec1_uni = TfidfVectorizer(ngram_range=(1, 1), lowercase=True)
vec2_uni = TfidfVectorizer(ngram_range=(2, 3), lowercase=True)

In [ ]:
vec1_train=vec1_uni.fit_transform(X_train)

In [ ]:
vec1_test=vec1_uni.transform(X_test)

In [ ]:
vec2_train = vec2_uni.fit_transform(X_train)

In [ ]:

vec2_test= vec2_uni.transform(X_test) # <-- Changed to .transform()

In [ ]:
from scipy.sparse import hstack, csr_matrix

In [ ]:
X_train = hstack([vec1_train, vec2_train])

In [ ]:
X_test = hstack([vec1_test, vec2_test])

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree

In [ ]:
svm_model_linear = SVC(kernel= 'linear', C = 1).fit(X_train, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Make predictions
svm_predictions = svm_model_linear.predict(X_test)

# --- 1️⃣ Confusion Matrix ---
cm = confusion_matrix(y_test, svm_predictions)
labels = ["other_cyberbullying", "religion", "age", "gender", "ethnicity"]

# Plot the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
plt.title("SVM Confusion Matrix")
plt.show()

# --- 2️⃣ Evaluation Metrics ---
accuracy = accuracy_score(y_test, svm_predictions)
print(f"Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, svm_predictions, target_names=labels))


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
dt_model = DecisionTreeClassifier(criterion='entropy', random_state=42)
dt_model.fit(X_train, y_train)
dt_predictions = dt_model.predict(X_test)

In [ ]:
labels = ["other_cyberbullying", "religion", "age", "gender", "ethnicity"]
cm = confusion_matrix(y_test, dt_predictions)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Oranges, xticks_rotation=45)
plt.title("Decision Tree Confusion Matrix")
plt.show()

# --- 4️⃣ Evaluation Metrics ---
accuracy = accuracy_score(y_test, dt_predictions)
print(f"Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, dt_predictions, target_names=labels))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)
labels = ["other_cyberbullying", "religion", "age", "gender", "ethnicity"]
cm = confusion_matrix(y_test, rf_predictions)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Greens, xticks_rotation=45)
plt.title("Random Forest Confusion Matrix")
plt.show()

# --- 4️⃣ Evaluation Metrics ---
accuracy = accuracy_score(y_test, rf_predictions)
print(f"Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
print(classification_report(y_test, rf_predictions, target_names=labels))


In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
from sklearn.ensemble import VotingClassifier

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# Assuming X_train and y_train are defined and loaded

clf1 = RandomForestClassifier(random_state=42)
clf2 = DecisionTreeClassifier(random_state=42)
# FIX: Set probability=True for SVC to enable soft voting
clf3 = SVC(kernel='linear', C=1, probability=True, random_state=42) 

estimators = [('RF', clf1), ('DT', clf2), ('SV', clf3)]

vc1 = VotingClassifier(estimators=estimators, voting='soft')

x = cross_val_score(vc1, X_train, y_train, cv=5, scoring='accuracy')

print(f"Mean Cross-Validation Accuracy: {np.round(np.mean(x), 2)}")

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

clf1 = RandomForestClassifier(random_state=42)
clf2 = DecisionTreeClassifier(random_state=42)
clf3 = SVC(kernel='linear', C=1, probability=True, random_state=42) 

estimators = [('RF', clf1), ('DT', clf2), ('SV', clf3)]

vc1 = VotingClassifier(estimators=estimators, voting='soft')


In [ ]:
# Train the VotingClassifier on the full training set
print("Fitting the Voting Classifier on training data...")
vc1.fit(X_train, y_train) 
print("Fitting complete.")

In [ ]:
# Use the trained model to predict the labels for the test data
y_pred = vc1.predict(X_test)

# You can now view the first few predictions
print("\nFirst 10 test predictions (y_pred):")
print(y_pred[:10])

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

vc_predictions = vc1.predict(X_test)
labels = ["other_cyberbullying", "religion", "age", "gender", "ethnicity"]
cm = confusion_matrix(y_test, vc_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(cmap=plt.cm.Purples, xticks_rotation=45)
plt.title("Voting Classifier Confusion Matrix")
plt.show()
accuracy = accuracy_score(y_test, vc_predictions)
print(f"Accuracy: {accuracy:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, vc_predictions, target_names=labels))

In [ ]:
df1.head(10)

In [ ]:
from sklearn.metrics import accuracy_score

# Assuming y_test (true test labels) is available
if 'y_test' in locals():
    test_accuracy = accuracy_score(y_test, y_pred)
    print(f"\nTest Set Accuracy: {np.round(test_accuracy, 4)}")